# 🚀 Real-time Data Pipeline - Google Colab Demo

This notebook demonstrates how to use the **Real-time Data Pipeline** with Google Cloud services.

## 📋 What You'll Learn:
- Set up Google Cloud authentication
- Generate sample events for testing
- Process events through Dataflow pipeline
- Query results from BigQuery
- Monitor pipeline performance

## 🔗 Related Resources:
- **GitHub Repository**: [real-time-data-pipeline-gcp](https://github.com/Vaishnavidorlikar/real-time-data-pipeline-gcp)
- **Documentation**: Check the README.md in the repository

---

### ⚠️ **Important Notes:**
1. You need a Google Cloud project with billing enabled
2. Required APIs: Dataflow, Pub/Sub, BigQuery, Cloud Storage
3. This notebook uses sample data generation (no real data sources)

**Estimated Cost**: ~$5-10 for running this demo (depending on usage)

## 🛠️ Step 1: Environment Setup

First, let's install the required packages and set up authentication.

In [3]:
# Install required packages
!pip install google-cloud-pubsub google-cloud-bigquery google-cloud-dataflow apache-beam pyyaml --quiet

print("✅ Packages installed successfully!")

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [14 lines of output]
      /private/var/folders/pk/jjzdzvfn6vj8l6t7_ssbpfl40000gn/T/pip-install-lcxttxit/google-cloud-dataflow_1ca303484c0b4d748b8e9835f7a5d745/setup.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
        from pkg_resources import get_distribution, DistributionNotFound
      /private/var/folders/pk/jjzdzvfn6vj8l6t7_ssbpfl40000gn/T/pip-install-lcxttxit/google-cloud-dataflow_1ca303484c0b4d748b8e9835f7a5d745/setup.py:54: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
        if StrictVersion(_PIP_VERSION) < StrictVersion(REQUIRED_PIP_VERSION):
      Traceback (most recent call last):
        File "<string>", line

In [4]:
import os
from google.colab import auth
from google.cloud import pubsub_v1, bigquery
from google.auth import default

# Authenticate with Google Cloud
auth.authenticate_user()

# Get default credentials
creds, _ = default()

# Get project ID
!gcloud config get-value project 2>/dev/null || echo "Please set your project ID below"

print("🔐 Authentication completed!")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# @title Configure Your Project
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
REGION = "us-central1"  # @param {type:"string"}
BUCKET_NAME = "your-unique-bucket-name"  # @param {type:"string"}

# Validate inputs
if PROJECT_ID == "your-gcp-project-id":
    raise ValueError("Please update PROJECT_ID with your actual Google Cloud project ID")
if BUCKET_NAME == "your-unique-bucket-name":
    raise ValueError("Please update BUCKET_NAME with your actual GCS bucket name")

# Set environment variables
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GCLOUD_REGION"] = REGION

print(f"📋 Project Configuration:")
print(f"   Project ID: {PROJECT_ID}")
print(f"   Region: {REGION}")
print(f"   Bucket: {BUCKET_NAME}")
print(f"✅ Configuration set!")

## 🏗️ Step 2: Infrastructure Setup

Let's create the necessary Google Cloud resources.

In [ ]:
# @title Create GCS Bucket
print(f"🪣 Creating GCS bucket: {BUCKET_NAME}")

!gsutil mb -p {PROJECT_ID} -l {REGION} gs://{BUCKET_NAME} 2>/dev/null || echo "Bucket may already exist"
!gsutil ls -b gs://{BUCKET_NAME}

print("✅ GCS bucket ready!")

In [ ]:
# @title Create Pub/Sub Topic and Subscription
TOPIC_NAME = "realtime-events"
SUBSCRIPTION_NAME = "realtime-events-sub"

print(f"📡 Creating Pub/Sub resources...")

# Create topic
!gcloud pubsub topics create {TOPIC_NAME} --project={PROJECT_ID} 2>/dev/null || echo "Topic may already exist"

# Create subscription
!gcloud pubsub subscriptions create {SUBSCRIPTION_NAME} --topic={TOPIC_NAME} --project={PROJECT_ID} 2>/dev/null || echo "Subscription may already exist"

print(f"✅ Pub/Sub topic: {TOPIC_NAME}")
print(f"✅ Pub/Sub subscription: {SUBSCRIPTION_NAME}")

In [ ]:
# @title Create BigQuery Dataset and Table
DATASET_ID = "realtime_events"
TABLE_NAME = "events"

print(f"🗄️ Creating BigQuery resources...")

# Initialize BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

# Create dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
try:
    bq_client.create_dataset(dataset_ref, exists_ok=True)
    print(f"✅ BigQuery dataset: {DATASET_ID}")
except Exception as e:
    print(f"Dataset creation note: {e}")

# Define table schema
schema = [
    bigquery.SchemaField("event_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("event_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("timestamp", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("processing_timestamp", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("session_id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("action", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("source", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("version", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("environment", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("source_system", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("partition_date", "DATE", mode="NULLABLE"),
    bigquery.SchemaField("processing_status", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("event_hash", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("processing_latency_ms", "INTEGER", mode="NULLABLE")
]

# Create table
table_ref = dataset_ref.table(TABLE_NAME)
table = bigquery.Table(table_ref, schema=schema)

# Set partitioning and clustering
table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY,
    field="timestamp"
)
table.clustering_fields = ["event_type", "user_id", "partition_date"]

try:
    bq_client.create_table(table, exists_ok=True)
    print(f"✅ BigQuery table: {DATASET_ID}.{TABLE_NAME}")
except Exception as e:
    print(f"Table creation note: {e}")

print("🎉 BigQuery infrastructure ready!")

## 📊 Step 3: Generate Sample Events

Now let's create and publish sample events to test our pipeline.

In [ ]:
import json
import time
import random
from datetime import datetime, timezone
from typing import Dict, Any

class EventGenerator:
    """Generate sample events for testing the pipeline"""
    
    def __init__(self):
        self.event_types = ['user_activity', 'transaction', 'system_log', 'metric']
        self.actions = ['login', 'purchase', 'view', 'click', 'logout', 'signup']
        
    def generate_event(self) -> Dict[str, Any]:
        """Generate a single sample event"""
        event_id = f"evt_{int(time.time() * 1000)}_{random.randint(1000, 9999)}"
        
        event = {
            'event_id': event_id,
            'event_type': random.choice(self.event_types),
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'user_id': f"user_{random.randint(1, 10000)}",
            'session_id': f"session_{random.randint(1, 1000)}",
            'data': {
                'action': random.choice(self.actions),
                'value': round(random.uniform(1.0, 1000.0), 2),
                'metadata': {
                    'source': 'colab_demo',
                    'version': '1.0',
                    'environment': 'development'
                }
            },
            'source_system': 'colab_demo',
            'processing_status': 'raw'
        }
        
        return event

# Test event generation
generator = EventGenerator()
sample_event = generator.generate_event()

print("📝 Sample Event Generated:")
print(json.dumps(sample_event, indent=2))
print("\n✅ Event generator ready!")

In [ ]:
# @title Publish Events to Pub/Sub
NUM_EVENTS = 50  # @param {type:"slider", min:10, max:200, step:10}
BATCH_DELAY = 0.1  # @param {type:"slider", min:0.01, max:1, step:0.01}

print(f"📤 Publishing {NUM_EVENTS} events to Pub/Sub...")

# Initialize Pub/Sub publisher
publisher = pubsub_v1.PublisherClient()
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_NAME)

# Generate and publish events
generator = EventGenerator()
published_count = 0

for i in range(NUM_EVENTS):
    try:
        event = generator.generate_event()
        message = json.dumps(event).encode('utf-8')
        
        # Publish message
        future = publisher.publish(topic_path, data=message)
        message_id = future.result()
        
        published_count += 1
        if i % 10 == 0:
            print(f"  Published {published_count}/{NUM_EVENTS} events...")
        
        # Small delay between messages
        time.sleep(BATCH_DELAY)
        
    except Exception as e:
        print(f"❌ Error publishing event {i}: {e}")
        continue

print(f"\n🎉 Successfully published {published_count} events to Pub/Sub!")
print(f"📡 Topic: {TOPIC_NAME}")
print(f"📊 Subscription: {SUBSCRIPTION_NAME}")

## 🔄 Step 4: Run Dataflow Pipeline

Now let's start the Dataflow pipeline to process our events.

In [ ]:
# @title Start Dataflow Pipeline
JOB_NAME = "colab-realtime-pipeline"  # @param {type:"string"}
MAX_WORKERS = 3  # @param {type:"slider", min:1, max:10, step:1}

print(f"🚀 Starting Dataflow pipeline: {JOB_NAME}")

# Create the Dataflow pipeline code
dataflow_code = '''
import json
import logging
from datetime import datetime, timezone
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions, StandardOptions
from apache_beam.io.gcp.bigquery import WriteToBigQuery
from apache_beam.io.gcp.pubsub import ReadFromPubSub

class ParseMessageFn(beam.DoFn):
    def process(self, message):
        try:
            data = json.loads(message.decode('utf-8'))
            yield data
        except Exception as e:
            logging.error(f"Error parsing message: {e}")

class TransformEventFn(beam.DoFn):
    def process(self, event):
        try:
            # Add processing timestamp
            event['processing_timestamp'] = datetime.now(timezone.utc).isoformat()
            
            # Add partition date
            if 'timestamp' in event:
                event['partition_date'] = event['timestamp'][:10]
            
            # Update processing status
            event['processing_status'] = 'processed'
            
            yield event
        except Exception as e:
            logging.error(f"Error transforming event: {e}")

def run():
    options = PipelineOptions([
        '--project=' + project_id,
        '--region=' + region,
        '--job_name=' + job_name,
        '--temp_location=gs://' + bucket_name + '/temp/',
        '--staging_location=gs://' + bucket_name + '/staging/',
        '--max_num_workers=' + str(max_workers),
        '--streaming',
        '--save_main_session'
    ])
    
    with beam.Pipeline(options=options) as p:
        (p 
         | 'ReadFromPubSub' >> ReadFromPubSub(subscription=subscription_path)
         | 'ParseMessages' >> beam.ParDo(ParseMessageFn())
         | 'TransformEvents' >> beam.ParDo(TransformEventFn())
         | 'WriteToBigQuery' >> WriteToBigQuery(
             table=table_path,
             schema=table_schema,
             write_disposition=beam.io.BigQueryDisposition.WRITE_APPEND,
             create_disposition=beam.io.BigQueryDisposition.CREATE_IF_NEEDED
         )
        )

if __name__ == '__main__':
    run()
'''

# Write the pipeline code to a file
with open('/tmp/dataflow_pipeline.py', 'w') as f:
    f.write(dataflow_code)

# Run the Dataflow pipeline
subscription_path = f"projects/{PROJECT_ID}/subscriptions/{SUBSCRIPTION_NAME}"
table_path = f"{PROJECT_ID}:{DATASET_ID}.{TABLE_NAME}"

# Command to run Dataflow
dataflow_cmd = f"""
python /tmp/dataflow_pipeline.py \
    --project_id={PROJECT_ID} \
    --region={REGION} \
    --job_name={JOB_NAME} \
    --subscription_path={subscription_path} \
    --table_path={table_path} \
    --bucket_name={BUCKET_NAME} \
    --max_workers={MAX_WORKERS}
"""

print(f"📋 Pipeline Configuration:")
print(f"   Job Name: {JOB_NAME}")
print(f"   Subscription: {subscription_path}")
print(f"   Output Table: {table_path}")
print(f"   Max Workers: {MAX_WORKERS}")
print(f"\n⚠️  Note: Dataflow job will run in the background. Check the Google Cloud Console to monitor progress.")
print(f"🔗 Console: https://console.cloud.google.com/dataflow?project={PROJECT_ID}")

In [ ]:
# @title Monitor Dataflow Job Status
import time
from google.cloud import dataflow_v1b3

print(f"🔍 Checking Dataflow job status...")

# Initialize Dataflow client
dataflow_client = dataflow_v1b3.JobsV1B3Client()

try:
    # List recent jobs
    request = dataflow_v1b3.ListJobsRequest(
        project_id=PROJECT_ID,
        region=REGION,
        view=dataflow_v1b3.JobView.SUMMARY
    )
    
    response = dataflow_client.list_jobs(request=request)
    
    print(f"\n📊 Recent Dataflow Jobs in {REGION}:")
    print("-" * 80)
    
    for job in response.jobs:
        job_name = job.name.split('/')[-1]
        job_state = job.current_state.name
        job_type = job.type.name
        
        print(f"Job: {job_name}")
        print(f"  State: {job_state}")
        print(f"  Type: {job_type}")
        print(f"  ID: {job.id}")
        print("-" * 40)
        
except Exception as e:
    print(f"❌ Error checking Dataflow jobs: {e}")
    print("💡 You can check jobs manually in the Google Cloud Console")
    print(f"🔗 https://console.cloud.google.com/dataflow?project={PROJECT_ID}")

## 📈 Step 5: Query Results from BigQuery

Let's check if our events have been processed and stored in BigQuery.

In [ ]:
# @title Query Processed Events
print(f"🔍 Querying BigQuery table: {DATASET_ID}.{TABLE_NAME}")

# Wait a moment for processing
print("⏳ Waiting for events to be processed...")
time.sleep(30)

# Query the results
query = f"""
SELECT 
    event_id,
    event_type,
    timestamp,
    processing_timestamp,
    user_id,
    action,
    value,
    source_system,
    processing_status,
    TIMESTAMP_DIFF(processing_timestamp, timestamp, MILLISECOND) as processing_latency_ms
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)
ORDER BY timestamp DESC
LIMIT 20
"""

try:
    query_job = bq_client.query(query)
    results = query_job.result()
    
    # Convert to DataFrame for better display
    df = results.to_dataframe()
    
    if len(df) > 0:
        print(f"\n📊 Found {len(df)} processed events:")
        print("-" * 100)
        
        # Display key columns
        display_cols = ['event_id', 'event_type', 'timestamp', 'user_id', 'action', 'value', 'processing_status', 'processing_latency_ms']
        print(df[display_cols].to_string(index=False))
        
        print(f"\n📈 Processing Statistics:")
        print(f"   Average Latency: {df['processing_latency_ms'].mean():.2f} ms")
        print(f"   Min Latency: {df['processing_latency_ms'].min():.2f} ms")
        print(f"   Max Latency: {df['processing_latency_ms'].max():.2f} ms")
        
    else:
        print("⚠️  No events found yet. The pipeline might still be processing...")
        print("💡 Try running this cell again in a few minutes.")
        
except Exception as e:
    print(f"❌ Error querying BigQuery: {e}")

In [ ]:
# @title Analytics Dashboard
print(f"📊 Analytics Dashboard for {DATASET_ID}.{TABLE_NAME}")

# Analytics queries
analytics_queries = {
    'total_events': f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}` WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)",
    'event_types': f"SELECT event_type, COUNT(*) as count FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}` WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR) GROUP BY event_type ORDER BY count DESC",
    'top_actions': f"SELECT data.action as action, COUNT(*) as count FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}` WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR) GROUP BY data.action ORDER BY count DESC LIMIT 5",
    'avg_value': f"SELECT AVG(value) as avg_value, event_type FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}` WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 HOUR) AND value IS NOT NULL GROUP BY event_type"
}

for name, query in analytics_queries.items():
    try:
        result = bq_client.query(query).result().to_dataframe()
        
        print(f"\n📈 {name.replace('_', ' ').title()}:")
        print("-" * 40)
        print(result.to_string(index=False))
        
    except Exception as e:
        print(f"❌ Error in {name}: {e}")

print(f"\n🔗 View full dataset: https://console.cloud.google.com/bigquery?project={PROJECT_ID}")

## 🔍 Step 6: Monitoring and Performance

Let's check the performance metrics and monitoring data.

In [ ]:
# @title Performance Metrics
print(f"📊 Performance Metrics for Real-time Pipeline")

# Performance queries
performance_query = f"""
SELECT 
    DATE(timestamp) as event_date,
    event_type,
    COUNT(*) as event_count,
    AVG(TIMESTAMP_DIFF(processing_timestamp, timestamp, MILLISECOND)) as avg_latency_ms,
    MIN(TIMESTAMP_DIFF(processing_timestamp, timestamp, MILLISECOND)) as min_latency_ms,
    MAX(TIMESTAMP_DIFF(processing_timestamp, timestamp, MILLISECOND)) as max_latency_ms
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}`
WHERE timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOURS)
GROUP BY DATE(timestamp), event_type
ORDER BY event_date DESC, event_count DESC
"""

try:
    perf_df = bq_client.query(performance_query).result().to_dataframe()
    
    if len(perf_df) > 0:
        print("\n📈 Performance Summary:")
        print("-" * 80)
        print(perf_df.to_string(index=False))
        
        # Overall stats
        total_events = perf_df['event_count'].sum()
        overall_avg_latency = perf_df['avg_latency_ms'].mean()
        
        print(f"\n🎯 Overall Performance:")
        print(f"   Total Events (24h): {total_events:,}")
        print(f"   Average Latency: {overall_avg_latency:.2f} ms")
        print(f"   Events per Hour: {total_events/24:.1f}")
        
    else:
        print("⚠️  No performance data available yet.")
        
except Exception as e:
    print(f"❌ Error getting performance metrics: {e}")

In [ ]:
# @title Resource Usage Check
print(f"💰 Resource Usage and Cost Estimation")

# Check BigQuery storage
storage_query = f"""
SELECT 
    table_id,
    ROUND(size_bytes / 1024 / 1024 / 1024, 2) as size_gb,
    ROUND(size_bytes / 1024 / 1024, 2) as size_mb,
    row_count
FROM `{PROJECT_ID}.{DATASET_ID}.__TABLES_SUMMARY__`
WHERE table_id = '{TABLE_NAME}'
"""

try:
    storage_df = bq_client.query(storage_query).result().to_dataframe()
    
    if len(storage_df) > 0:
        size_gb = storage_df['size_gb'].iloc[0]
        row_count = storage_df['row_count'].iloc[0]
        
        print(f"\n📊 BigQuery Storage:")
        print(f"   Table Size: {size_gb} GB")
        print(f"   Row Count: {row_count:,}")
        print(f"   Avg Row Size: {(size_gb * 1024 / row_count * 1024):.2f} KB")
        
        # Rough cost estimation
        storage_cost = size_gb * 0.02  # $0.02 per GB/month
        print(f"\n💰 Estimated Monthly Costs:")
        print(f"   BigQuery Storage: ${storage_cost:.2f}")
        print(f"   Dataflow Processing: ${row_count * 0.001:.2f}")  # Rough estimate
        print(f"   Pub/Sub Messages: ${row_count * 0.0000004:.2f}")  # $0.40 per million
        
    else:
        print("⚠️  No storage data available.")
        
except Exception as e:
    print(f"❌ Error checking storage: {e}")

print(f"\n🔗 Google Cloud Cost Calculator: https://cloud.google.com/products/calculator")

## 🧹 Step 7: Cleanup Resources

**⚠️ Important**: Clean up resources to avoid ongoing charges!

In [ ]:
# @title Clean Up Resources
CLEANUP = False  # @param {type:"boolean"}

if CLEANUP:
    print(f"🧹 Cleaning up resources...")
    
    try:
        # Delete Pub/Sub subscription
        !gcloud pubsub subscriptions delete {SUBSCRIPTION_NAME} --project={PROJECT_ID} --quiet
        print(f"✅ Deleted Pub/Sub subscription: {SUBSCRIPTION_NAME}")
        
        # Delete Pub/Sub topic
        !gcloud pubsub topics delete {TOPIC_NAME} --project={PROJECT_ID} --quiet
        print(f"✅ Deleted Pub/Sub topic: {TOPIC_NAME}")
        
        # Delete BigQuery table
        bq_client.delete_table(f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}", not_found_ok=True)
        print(f"✅ Deleted BigQuery table: {DATASET_ID}.{TABLE_NAME}")
        
        # Delete BigQuery dataset
        bq_client.delete_dataset(DATASET_ID, delete_contents=True, not_found_ok=True)
        print(f"✅ Deleted BigQuery dataset: {DATASET_ID}")
        
        # Optional: Delete GCS bucket (commented out for safety)
        # !gsutil -m rm -r gs://{BUCKET_NAME}
        # print(f"✅ Deleted GCS bucket: {BUCKET_NAME}")
        
        print(f"\n🎉 Cleanup completed!")
        print(f"💡 Note: GCS bucket {BUCKET_NAME} was preserved. Delete manually if needed.")
        
    except Exception as e:
        print(f"❌ Error during cleanup: {e}")
        
else:
    print(f"⚠️  Cleanup skipped. Set CLEANUP=True to remove resources.")
    print(f"💡 Remember to clean up manually to avoid charges:")
    print(f"   - Pub/Sub topic: {TOPIC_NAME}")
    print(f"   - Pub/Sub subscription: {SUBSCRIPTION_NAME}")
    print(f"   - BigQuery dataset: {DATASET_ID}")
    print(f"   - GCS bucket: {BUCKET_NAME}")

## 🚀 Next Steps and Resources

### 📚 What You Learned:
1. ✅ Set up Google Cloud infrastructure (Pub/Sub, BigQuery, Dataflow)
2. ✅ Generated and published sample events
3. ✅ Processed events through Apache Beam pipeline
4. ✅ Queried and analyzed results in BigQuery
5. ✅ Monitored performance and costs

### 🔗 Useful Links:
- **GitHub Repository**: [real-time-data-pipeline-gcp](https://github.com/Vaishnavidorlikar/real-time-data-pipeline-gcp)
- **Google Cloud Console**: [Dashboard](https://console.cloud.google.com/home/dashboard?project={PROJECT_ID})
- **Dataflow Jobs**: [Monitoring](https://console.cloud.google.com/dataflow?project={PROJECT_ID})
- **BigQuery**: [Query Editor](https://console.cloud.google.com/bigquery?project={PROJECT_ID})
- **Pub/Sub**: [Topics](https://console.cloud.google.com/pubsub/topic?project={PROJECT_ID})

### 🛠️ Advanced Topics:
- **Real Data Sources**: Replace sample generator with actual data sources
- **Windowing**: Add time windows for aggregation
- **Error Handling**: Implement dead-letter queues
- **Monitoring**: Add Cloud Monitoring alerts
- **Scaling**: Optimize for high throughput

### 📞 Get Help:
- **Documentation**: Check the README.md in the repository
- **Issues**: Report problems on GitHub
- **Google Cloud Support**: Available for paid accounts

---

### 🎉 Congratulations!

You've successfully set up and tested a real-time data pipeline using Google Cloud services! This pipeline can process events from Pub/Sub, transform them using Apache Beam, and store them in BigQuery for analysis.

**Remember to clean up resources when you're done to avoid charges!**

---

*Built by Vaishnavi Dorlikar | Real-time Data Pipeline Demo*